# Pipeline Stage 1 — Ground Filtering Only

This notebook contains **only the filtering half** of the v7 pipeline. It runs
Patchwork++ on a single LiDAR frame (or a multi-LiDAR MCAP frame), optionally
applies an extra **eigenvalue + flatness post-filter** (the GLE indicators from
the Patchwork++ paper, applied as a *stricter* second pass), and saves the
surviving ground points to a `.bin` file.

That `.bin` is then the *input* to **Notebook 2** (`compute_slope_curvature.ipynb`),
which runs the slope/curvature calculations. Splitting the notebooks lets us
mix and match:

| Filter notebook produces this `.bin` | Calculation notebook uses these methods |
|---|---|
| Patchwork++ only | PCA · RANSAC · height-derivative |
| Patchwork++ + GLE post-filter | PCA · RANSAC · height-derivative |
| RANSAC pipeline (collaborator's notebook) | PCA · RANSAC · height-derivative |

So the filtration choice and the local-fit choice become **two independent
variables** the BEP report can sweep separately.

### Analogy — the cartographer revisits the park

Stage 1 of the v7 notebook was: *throw away everything that isn't ground.*
This notebook adds an optional extra step: *after Patchwork++ has done its
best, look at every patch of "ground" and double-check it really is flat-ish
and roughly horizontal.* Patches that fail get dropped too. It's the second
pair of eyes that the cartographer asks a colleague for before they print the map.

> **Be careful.** This filter rejects patches with *high curvature*, which is
> precisely what we're trying to *measure* downstream. Use it lightly. The
> default thresholds are intentionally lenient, and the BEP report should
> defend whatever values you end up using.

In [ ]:
# Run once if you don't have these:
# !pip install numpy matplotlib pypatchworkpp
# !pip install rosbags          # only needed for MCAP support

import os
import numpy as np
import matplotlib.pyplot as plt
import pypatchworkpp

np.set_printoptions(suppress=True, precision=4)


# ==========================================================
# WORKFLOW SELECTION
# ----------------------------------------------------------
#   "bin"  -> a nuScenes-format .bin file
#   "mcap" -> a frame from a ROS bag (.mcap file, multi-LiDAR aware)
# ==========================================================
WORKFLOW = "mcap"          # "bin"  or  "mcap"


# ==========================================================
# OUTPUT path -- where Notebook 2 will read its input from
# ----------------------------------------------------------
# If APPLY_EIGEN_FLATNESS_FILTER is True the suffix
# "_pw_gle.bin" is added so different runs don't overwrite
# each other and you can compare them in Notebook 2.
# ==========================================================
EXPORT_GROUND   = True
GROUND_BIN_DIR  = r"C:\Users\JRepa\OneDrive - Delft University of Technology\Documenten\02. TU Delft\2025-2026\BEP\Data\Ground points"
GROUND_BIN_BASE = "ground_points"   # suffix is appended automatically below


# ==========================================================
# OPTIONAL POST-FILTER (Patchwork++ GLE-style)
# ----------------------------------------------------------
# After Patchwork++ runs, lay a coarse grid over the ground
# points, fit a PCA plane in each cell, and reject the whole
# cell if either:
#   - flatness  sigma = lambda3 / (lambda1+lambda2+lambda3)
#                exceeds FLATNESS_THRESHOLD     (cell too "fluffy")
#   - uprightness |n_z|
#                falls below UPRIGHTNESS_THRESHOLD (normal too tilted)
#
# These are the same indicators Patchwork++ uses internally
# (paper Sec. III-D, Eq. of sigma_n). Using them again as a
# post-filter is a strictness knob: it removes the borderline
# cells that Patchwork++'s adaptive thresholds happened to
# accept this frame.
#
# WARNING: on a curvature-focused pipeline this can throw out
# the very signal you want. Keep the flatness threshold fairly
# loose (>= 0.02) unless you have a good reason.
# ==========================================================
APPLY_EIGEN_FLATNESS_FILTER = False   # toggle the optional pass on/off

POST_CELL_SIZE          = 1.0     # [m]
POST_MIN_PTS_PER_CELL   = 12      # cells with fewer points are kept as-is
FLATNESS_THRESHOLD      = 0.03    # reject cell if sigma > this
UPRIGHTNESS_THRESHOLD   = 0.85    # reject cell if |n_z| < this  (~31 deg tilt)


# ==========================================================
# Frame selection from MCAP
# ==========================================================
MCAP_FRAME_INDEX     = None
MCAP_FRAME_TIMESTAMP = 1777639270.466376996
MCAP_TOPIC           = None      # None = use all LiDAR topics + calibrate


# ==========================================================
# Sensor configuration (single source of truth)
# ==========================================================
if WORKFLOW == "bin":
    _sensor_height = 1.84
else:
    _sensor_height = 0.9

SENSOR_CONFIG = {
    "sensor_height": _sensor_height,
    "min_range":  1.0,
    "max_range": 50.0,
    "verbose":   False,
}


# ==========================================================
# I/O paths
# ==========================================================
BIN_PATH  = r"C:\Users\JRepa\OneDrive - Delft University of Technology\Documenten\02. TU Delft\2025-2026\BEP\Data\Pointclouds curvature\pointcloud.bin"
MCAP_PATH = r"D:\Data_gathered\2026_05_01\Rosbag\14_40_00\rosbag\rosbag_0.mcap"

LIDAR_TOPICS = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_R",
    "/rslidar/helios_L",
]

BASE_FRAME = "base_link"


## 1. Loaders

Two formats are supported:

* **`.bin`** — nuScenes layout: `float32` array reshaped to `(N, 5)` → `[x, y, z, intensity, ring]`.
* **`.mcap`** — a ROS bag containing one or more `sensor_msgs/PointCloud2` topics.
  When multiple LiDAR topics are loaded, each is run through Patchwork++
  *separately in its own sensor frame* (so `sensor_height` is correct), and only
  *afterwards* are the results moved into `base_link` via `/tf_static`.

Both paths produce an `(N, 3)` `float64` XYZ array.

In [ ]:
# ------------------------------------------------------------
# Loader 1: nuScenes-format .bin file
# ------------------------------------------------------------
def load_nuscenes_bin(path):
    """Load (N, 3) XYZ from a nuScenes-format .bin file."""
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 5)
    return pts[:, :3].astype(np.float64)


# ------------------------------------------------------------
# Loader 2: a single frame from a ROS .mcap bag
# ------------------------------------------------------------
_PF_DTYPE = {1: ("i1", 1), 2: ("u1", 1), 3: ("i2", 2), 4: ("u2", 2),
             5: ("i4", 4), 6: ("u4", 4), 7: ("f4", 4), 8: ("f8", 8)}


def _pointcloud2_to_xyz(msg):
    """Extract (xyz, frame_id) from a PointCloud2 message."""
    field_map = {f.name: f for f in msg.fields}
    missing = [k for k in ("x", "y", "z") if k not in field_map]
    if missing:
        raise RuntimeError(f"PointCloud2 missing {missing}. Have: {list(field_map)}")

    n = msg.width * msg.height
    raw = np.frombuffer(msg.data, dtype=np.uint8).reshape(n, msg.point_step)

    xyz = np.empty((n, 3), dtype=np.float64)
    for i, name in enumerate(("x", "y", "z")):
        f = field_map[name]
        dtype_str, size = _PF_DTYPE[f.datatype]
        col_bytes = raw[:, f.offset:f.offset + size].copy()
        xyz[:, i] = col_bytes.view(np.dtype(dtype_str)).flatten().astype(np.float64)

    valid = np.isfinite(xyz).all(axis=1)
    return xyz[valid], msg.header.frame_id


# ============================================================
# EXTRINSIC CALIBRATION helpers (/tf_static)
# ------------------------------------------------------------
# Calibration is applied AFTER Patchwork++ -- otherwise the
# ground z values shift and Patchwork++'s sensor_height
# assumption breaks.
# ============================================================
_TF_TREE_CACHE = {}


def _quat_to_R(x, y, z, w):
    n = x*x + y*y + z*z + w*w
    if n < 1e-12:
        return np.eye(3)
    s = 2.0 / n
    return np.array([
        [1 - s*(y*y + z*z),     s*(x*y - z*w),     s*(x*z + y*w)],
        [    s*(x*y + z*w), 1 - s*(x*x + z*z),     s*(y*z - x*w)],
        [    s*(x*z - y*w),     s*(y*z + x*w), 1 - s*(x*x + y*y)],
    ])


def _build_tf_tree(mcap_path):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    if mcap_path in _TF_TREE_CACHE:
        return _TF_TREE_CACHE[mcap_path]

    tree = {}
    with AnyReader([Path(mcap_path)]) as reader:
        wanted = [c for c in reader.connections if c.topic == "/tf_static"]
        if not wanted:
            print("[tf] WARNING: /tf_static not found -- no calibration available.")
            _TF_TREE_CACHE[mcap_path] = tree
            return tree
        for conn, ts, raw in reader.messages(connections=wanted):
            msg = reader.deserialize(raw, conn.msgtype)
            for tf in msg.transforms:
                parent = tf.header.frame_id
                child  = tf.child_frame_id
                t = np.array([tf.transform.translation.x,
                              tf.transform.translation.y,
                              tf.transform.translation.z])
                q = tf.transform.rotation
                R = _quat_to_R(q.x, q.y, q.z, q.w)
                tree[child] = (parent, R, t)
    print(f"[tf] Parsed {len(tree)} static transforms. Frames: {sorted(tree.keys())}")
    _TF_TREE_CACHE[mcap_path] = tree
    return tree


def _resolve_to_base(tree, child_frame, base_frame="base_link", max_depth=10):
    R_acc = np.eye(3); t_acc = np.zeros(3); cur = child_frame
    for _ in range(max_depth):
        if cur == base_frame:
            return R_acc, t_acc
        if cur not in tree:
            raise RuntimeError(f"'{cur}' not in /tf_static -- cannot reach '{base_frame}'.")
        parent, R, t = tree[cur]
        t_acc = R @ t_acc + t
        R_acc = R @ R_acc
        cur = parent
    raise RuntimeError(f"TF chain too deep from '{child_frame}'.")


def _apply_tf(xyz, R, t):
    return xyz @ R.T + t


# ============================================================
# Fast MCAP loader -- caches per-topic clouds for post-PW++
# calibration in _LAST_LOAD_CACHE.
# ============================================================
_MCAP_INDEX_CACHE = {}
_LAST_LOAD_CACHE  = {}


def _build_mcap_index(path):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    if path in _MCAP_INDEX_CACHE:
        return _MCAP_INDEX_CACHE[path]

    print("[mcap] Indexing bag (one-time)... ", end="", flush=True)
    index = {}
    with AnyReader([Path(path)]) as reader:
        pc2_conns = [c for c in reader.connections
                     if c.msgtype == "sensor_msgs/msg/PointCloud2"]
        for c in pc2_conns:
            index[c.topic] = []
        for conn, ts, raw in reader.messages(connections=pc2_conns):
            index[conn.topic].append(ts)
    for topic in index:
        index[topic] = np.asarray(index[topic], dtype=np.int64)
    _MCAP_INDEX_CACHE[path] = {"index": index}
    print(f"done. Topics: {list(index.keys())}")
    return _MCAP_INDEX_CACHE[path]


def _resolve_frame_time(path, target_idx, target_sec, ref_topic):
    cache = _build_mcap_index(path)
    ts_list = cache["index"][ref_topic]
    if target_sec is not None:
        target_ns = int(target_sec * 1e9)
        i = int(np.argmin(np.abs(ts_list - target_ns)))
        diff_s = abs(ts_list[i] - target_ns) / 1e9
        print(f"[mcap] Timestamp {target_sec:.3f} -> frame {i} on {ref_topic} "
              f"(off by {diff_s:.4f} s)")
        return ts_list[i], i
    if target_idx is not None:
        return ts_list[target_idx], target_idx
    raise ValueError("Need MCAP_FRAME_INDEX or MCAP_FRAME_TIMESTAMP.")


def load_mcap_frame_fast(path, topics, target_idx=None, target_sec=None,
                         tol_sec=0.05):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    ref_topic = topics[0]
    target_ns, ref_idx = _resolve_frame_time(path, target_idx, target_sec, ref_topic)

    tol_ns = int(tol_sec * 1e9)
    start_ns = target_ns - tol_ns
    end_ns   = target_ns + tol_ns

    out = {t: None for t in topics}

    with AnyReader([Path(path)]) as reader:
        wanted = [c for c in reader.connections if c.topic in set(topics)]
        for conn, ts, raw in reader.messages(connections=wanted,
                                             start=start_ns,
                                             stop=end_ns + 1):
            if out[conn.topic] is None:
                msg = reader.deserialize(raw, conn.msgtype)
                xyz, frame_id = _pointcloud2_to_xyz(msg)
                out[conn.topic] = (xyz, frame_id)
                if all(v is not None for v in out.values()):
                    break

    _LAST_LOAD_CACHE.clear()

    parts = []
    for t in topics:
        if out[t] is None:
            print(f"  [warn] {t}: no message within +/-{tol_sec*1000:.0f} ms")
        else:
            xyz, fid = out[t]
            _LAST_LOAD_CACHE[t] = (xyz, fid)
            print(f"  {t}: {len(xyz):,} pts  (frame: {fid})")
            parts.append(xyz)

    if not parts:
        raise RuntimeError("No LiDAR frames matched.")

    merged = np.concatenate(parts, axis=0)
    print(f"  Merged total: {len(merged):,} points (raw, uncalibrated)")
    return merged


def load_mcap_frame(path, topic=None, frame_index=0, list_topics=False):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    cache = _build_mcap_index(path)
    if list_topics:
        print("PointCloud2 topics in this bag:")
        for t, ts in cache["index"].items():
            print(f"  {t}  ({len(ts)} messages)")
        return None

    if topic is None:
        topic = next(iter(cache["index"].keys()))
        print(f"[mcap] Auto-selected topic: {topic}")

    target_ns = cache["index"][topic][frame_index]
    with AnyReader([Path(path)]) as reader:
        wanted = [c for c in reader.connections if c.topic == topic]
        for conn, ts, raw in reader.messages(connections=wanted,
                                             start=target_ns,
                                             stop=target_ns + 1):
            msg = reader.deserialize(raw, conn.msgtype)
            xyz, fid = _pointcloud2_to_xyz(msg)
            print(f"[mcap] Loaded frame {frame_index} from {topic} (frame: {fid})")
            return xyz
    raise RuntimeError("Frame not found.")


def load_frame():
    if WORKFLOW == "bin":
        return load_nuscenes_bin(BIN_PATH)
    elif WORKFLOW == "mcap":
        if MCAP_TOPIC is not None:
            cache = _build_mcap_index(MCAP_PATH)
            ts_list = cache["index"][MCAP_TOPIC]
            if MCAP_FRAME_TIMESTAMP is not None:
                idx = int(np.argmin(np.abs(ts_list - int(MCAP_FRAME_TIMESTAMP*1e9))))
            else:
                idx = MCAP_FRAME_INDEX
            return load_mcap_frame(MCAP_PATH, topic=MCAP_TOPIC, frame_index=idx)
        return load_mcap_frame_fast(
            MCAP_PATH, topics=LIDAR_TOPICS,
            target_idx=MCAP_FRAME_INDEX,
            target_sec=MCAP_FRAME_TIMESTAMP,
        )
    else:
        raise ValueError(f"Unknown WORKFLOW '{WORKFLOW}'. Use 'bin' or 'mcap'.")


points = load_frame()
print(f"\nWorkflow:  {WORKFLOW}")
print(f"Loaded     {len(points):,} points")
print(f"X range:   [{points[:,0].min():.1f}, {points[:,0].max():.1f}] m")
print(f"Y range:   [{points[:,1].min():.1f}, {points[:,1].max():.1f}] m")
print(f"Z range:   [{points[:,2].min():.1f}, {points[:,2].max():.1f}] m")

## 2. Quick look at the raw cloud

Sanity check before doing anything clever.

In [ ]:
def bev_plot(points, title="BEV", lim=40, c=None, cmap="viridis",
             cbar_label=None, ax=None, s=1):
    """Top-down scatter of a point cloud. If `c` is None, colour by height."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    if c is None:
        c = points[:, 2]
    sc = ax.scatter(points[:, 0], points[:, 1], c=c, s=s, cmap=cmap)
    ax.set_xlabel("X forward [m]")
    ax.set_ylabel("Y left [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "Z [m]", shrink=0.8)
    return ax

# --- Side view (X-Z): shows slopes, hills, and the ground profile ---
def side_plot(points, title="Side view", lim=40, c=None, cmap="viridis",
              cbar_label=None, ax=None, s=1):
    """Side projection: X (forward) vs Z (height)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    if c is None:
        c = points[:, 1]  # colour by Y (left-right) to add depth
    sc = ax.scatter(points[:, 0], points[:, 2], c=c, s=s, cmap=cmap)
    ax.set_xlabel("X forward [m]")
    ax.set_ylabel("Z height [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-5, 10)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "Y [m]", shrink=0.8)
    return ax


# --- Front view (Y-Z): shows the road cross-section ---
def front_plot(points, title="Front view", lim=40, c=None, cmap="viridis",
               cbar_label=None, ax=None, s=1):
    """Front projection: Y (left-right) vs Z (height)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    if c is None:
        c = points[:, 0]  # colour by X (distance ahead)
    sc = ax.scatter(points[:, 1], points[:, 2], c=c, s=s, cmap=cmap)
    ax.set_xlabel("Y left [m]")
    ax.set_ylabel("Z height [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-5, 10)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "X [m]", shrink=0.8)
    return ax


# --- Show all three views together ---
fig, axes = plt.subplots(3, 1, figsize=(12, 16))

# BEV (top-down)
bev_plot(points, title="Bird's-eye view — coloured by height", ax=axes[0])

# Side view (X-Z)
side_plot(points, title="Side view — coloured by Y (left-right)", ax=axes[1])

# Front view (Y-Z)
front_plot(points, title="Front view — coloured by X (forward)", ax=axes[2])

plt.tight_layout()
plt.show()

## 3. Stage 1a — Patchwork++ ground segmentation

We use the authors' official C++ implementation via `pypatchworkpp`.

**One-paragraph recap.** Patchwork++ splits the ground around the sensor into
**concentric zones**, each zone into **angular wedges** (CZM). Inside each
wedge it runs **R-GPF**: sort by height, take the lowest slice as seeds, fit
a plane through them with PCA, reject wedges whose plane is too vertical or
whose seeds are too spread in z, label everything within a distance threshold
of the plane as ground. The `++` adds Adaptive Ground Likelihood Estimation
(A-GLE), Temporal Ground Revert, and Reflected-Noise Removal.

> **Analogy.** Each wedge is a small patch of parking lot. You stand in it,
> squint at the ~20 lowest points, lay a ruler across them, and mark every
> other point within an inch of the ruler as floor. Everything else is obstacle.

In [ ]:
# ---------------------------------------------------------------
# PCA helper -- used both by the Patchwork++ wrapper (for the
# fallback path) and by the optional eigenvalue+flatness post-filter.
# ---------------------------------------------------------------
def _pca_plane(pts):
    """Fit a plane through `pts` with PCA.

    Returns
    -------
    n        : (3,) unit normal, oriented so n_z >= 0
    c        : (3,) centroid
    eigvals  : (3,) eigenvalues, sorted DESCENDING (lam1 >= lam2 >= lam3)
    """
    pts = pts.astype(np.float64, copy=False)
    c = pts.mean(axis=0)
    cov = np.cov((pts - c).T)
    w, V = np.linalg.eigh(cov)          # ascending
    eigvals = w[::-1]                   # descending
    n = V[:, 0]                         # smallest eigval -> normal
    if n[2] < 0:
        n = -n
    return n, c, eigvals


def build_patchworkpp(config):
    params = pypatchworkpp.Parameters()
    params.sensor_height = float(config["sensor_height"])
    params.min_range     = float(config["min_range"])
    params.max_range     = float(config["max_range"])
    params.verbose       = bool(config.get("verbose", False))
    return pypatchworkpp.patchworkpp(params)


PatchworkPP = build_patchworkpp(SENSOR_CONFIG)
print(f"Patchwork++ configured with sensor_height = {SENSOR_CONFIG['sensor_height']} m, "
      f"range [{SENSOR_CONFIG['min_range']}, {SENSOR_CONFIG['max_range']}] m")


def segment_ground(points):
    """Run Patchwork++. Returns (ground, nonground) as (M, 3) arrays."""
    if points.shape[1] == 3:
        pts4 = np.hstack([points, np.zeros((len(points), 1))]).astype(np.float32)
    else:
        pts4 = points[:, :4].astype(np.float32)
    PatchworkPP.estimateGround(pts4)
    ground    = np.asarray(PatchworkPP.getGround())[:, :3]
    nonground = np.asarray(PatchworkPP.getNonground())[:, :3]
    return ground, nonground

In [ ]:
# ============================================================
# Run Patchwork++  --  per-sensor when MCAP has multiple LiDARs,
# then apply /tf_static calibration into base_link.
# ============================================================

if WORKFLOW == "mcap" and MCAP_TOPIC is None and _LAST_LOAD_CACHE:
    # --- Multi-topic MCAP: per-sensor segmentation + calibration ---
    tree = _build_tf_tree(MCAP_PATH)
    have_tf = bool(tree)

    ground_parts, nonground_parts = [], []

    for topic, (raw_xyz, frame_id) in _LAST_LOAD_CACHE.items():
        g, ng = segment_ground(raw_xyz)

        # Apply extrinsic calibration (sensor frame -> base_link)
        if have_tf:
            try:
                R, t = _resolve_to_base(tree, frame_id, BASE_FRAME)
                g  = _apply_tf(g,  R, t)
                ng = _apply_tf(ng, R, t)
                print(f"  {topic}: {len(g):,} ground, {len(ng):,} obstacle  "
                      f"({frame_id} -> {BASE_FRAME})")
            except RuntimeError as e:
                print(f"  [warn] {topic}: {e} -- using raw frame")
        else:
            print(f"  {topic}: {len(g):,} ground, {len(ng):,} obstacle  (uncalibrated)")

        ground_parts.append(g)
        nonground_parts.append(ng)

    ground    = np.concatenate(ground_parts,    axis=0)
    nonground = np.concatenate(nonground_parts, axis=0)
    print(f"\n  Calibrated total: {len(ground):,} ground, "
          f"{len(nonground):,} obstacle")

else:
    # --- Single-topic MCAP or .bin: no per-sensor split needed ---
    ground, nonground = segment_ground(points)

print(f"\nGround:    {len(ground):,} points")
print(f"Obstacles: {len(nonground):,} points")
print(f"Ground fraction: {100 * len(ground) / (len(ground) + len(nonground)):.1f} %")


# ============================================================
# Visualize ground vs obstacle
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].scatter(nonground[:, 0], nonground[:, 1], s=1, c="crimson", label="Obstacle")
axes[0].scatter(ground[:, 0],    ground[:, 1],    s=1, c="forestgreen", label="Ground")
axes[0].set_title("Stage 1 result — ground vs obstacle (BEV)")
axes[0].set_xlabel("X [m]"); axes[0].set_ylabel("Y [m]")
axes[0].set_xlim(-40, 40); axes[0].set_ylim(-40, 40)
axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3); axes[0].legend(markerscale=5)

sc = axes[1].scatter(ground[:, 0], ground[:, 1], c=ground[:, 2], s=1, cmap="terrain")
plt.colorbar(sc, ax=axes[1], label="Z [m]", shrink=0.8)
axes[1].set_title("Ground points only — coloured by height")
axes[1].set_xlabel("X [m]"); axes[1].set_ylabel("Y [m]")
axes[1].set_xlim(-40, 40); axes[1].set_ylim(-40, 40)
axes[1].set_aspect("equal"); axes[1].grid(True, lw=0.3)
plt.tight_layout(); plt.show()

# --- Side view ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].scatter(nonground[:, 0], nonground[:, 2], s=1, c="crimson", label="Obstacle")
axes[0].scatter(ground[:, 0],    ground[:, 2],    s=1, c="forestgreen", label="Ground")
axes[0].set_title("Stage 1 result — ground vs obstacle (Side view)")
axes[0].set_xlabel("X forward [m]"); axes[0].set_ylabel("Z height [m]")
axes[0].set_xlim(-40, 40); axes[0].set_ylim(-5, 10)
axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3); axes[0].legend(markerscale=5)

sc = axes[1].scatter(ground[:, 0], ground[:, 2], c=ground[:, 2], s=1, cmap="terrain")
plt.colorbar(sc, ax=axes[1], label="Z [m]", shrink=0.8)
axes[1].set_title("Ground points only — side view coloured by height")
axes[1].set_xlabel("X forward [m]"); axes[1].set_ylabel("Z height [m]")
axes[1].set_xlim(-40, 40); axes[1].set_ylim(-5, 10)
axes[1].set_aspect("equal"); axes[1].grid(True, lw=0.3)
plt.tight_layout(); plt.show()

# --- Front view ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].scatter(nonground[:, 1], nonground[:, 2], s=1, c="crimson", label="Obstacle")
axes[0].scatter(ground[:, 1],    ground[:, 2],    s=1, c="forestgreen", label="Ground")
axes[0].set_title("Stage 1 result — ground vs obstacle (Front view)")
axes[0].set_xlabel("Y left [m]"); axes[0].set_ylabel("Z height [m]")
axes[0].set_xlim(-40, 40); axes[0].set_ylim(-5, 10)
axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3); axes[0].legend(markerscale=5)

sc = axes[1].scatter(ground[:, 1], ground[:, 2], c=ground[:, 2], s=1, cmap="terrain")
plt.colorbar(sc, ax=axes[1], label="Z [m]", shrink=0.8)
axes[1].set_title("Ground points only — front view coloured by height")
axes[1].set_xlabel("Y left [m]"); axes[1].set_ylabel("Z height [m]")
axes[1].set_xlim(-40, 40); axes[1].set_ylim(-5, 10)
axes[1].set_aspect("equal"); axes[1].grid(True, lw=0.3)
plt.tight_layout(); plt.show()

# --- 3D interactive ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

step_g  = max(1, len(ground) // 20000)
step_ng = max(1, len(nonground) // 10000)

ax.scatter(nonground[::step_ng, 0], nonground[::step_ng, 1], nonground[::step_ng, 2],
           s=0.5, c="crimson", alpha=0.4, label="Obstacle")
ax.scatter(ground[::step_g, 0], ground[::step_g, 1], ground[::step_g, 2],
           s=0.5, c="forestgreen", alpha=0.6, label="Ground")
ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]"); ax.set_zlabel("Z [m]")
ax.set_title("3D view after Patchwork++ (drag to rotate)")
ax.set_xlim(-40, 40); ax.set_ylim(-40, 40); ax.set_zlim(-5, 10)
ax.legend(markerscale=10)
plt.show()

## 4. Stage 1b *(optional)* — Eigenvalue + flatness post-filter

If `APPLY_EIGEN_FLATNESS_FILTER` is `True`, every cell on a coarse grid over
the ground points is examined again and possibly *rejected*. Two indicators
from the Patchwork++ paper (Sec. III-D) are reused:

1. **Flatness (surface variation)**

$$\sigma_n \;=\; \frac{\lambda_3}{\lambda_1+\lambda_2+\lambda_3}$$

   measures how "thick" the cell is perpendicular to the best-fit plane.
   Flat asphalt → $\sigma_n \approx 0$. Bushes, kerbs, vegetation → $\sigma_n$ grows.
   We **reject** the cell if $\sigma_n >$ `FLATNESS_THRESHOLD`.

2. **Uprightness**

$$\text{uprightness} \;=\; |n_z|$$

   is how vertical the plane normal is. A real ground patch has $|n_z|$ close
   to 1; a vertical wall mistaken for ground has $|n_z|$ close to 0.
   We **reject** the cell if $|n_z| <$ `UPRIGHTNESS_THRESHOLD`.

### Why a *second* pass when Patchwork++ already does this?

Patchwork++'s A-GLE thresholds are *adaptive per ring* and chosen to maximise
recall (catch as much ground as possible). Some borderline patches sneak
through. A static post-filter at our chosen strictness reproducibly removes
those. Crucially — by separating it from Patchwork++ — we can **report ablations**
in the BEP: filtration with vs without the post-pass, on the same input frame.

### Caveat (and why this is a *toggle*, not the default)

For our project the goal is to *measure curvature*, and curvature = points
that don't lie in a single plane = high $\sigma_n$. Use the filter sparingly:
the defaults (`flatness ≤ 0.03`, `uprightness ≥ 0.85`) are deliberately loose
to drop only the obvious mis-classifications (low ledges, kerbs, sloped
shoulders) without sacrificing genuine hill curvature. Tighten only with eyes wide open.

In [ ]:
def eigen_flatness_filter(ground_points,
                          cell_size=1.0,
                          min_pts_per_cell=12,
                          flatness_thresh=0.03,
                          uprightness_thresh=0.85,
                          x_range=(-40, 40),
                          y_range=(-40, 40)):
    """Optional post-filter on Patchwork++ ground.

    Lay a uniform Cartesian grid. For each cell with at least
    `min_pts_per_cell` points, compute PCA -> (normal, eigenvalues).
    Reject the cell (drop all its points) if either:
        - sigma = lambda3 / (lam1+lam2+lam3) > flatness_thresh
        - |n_z| < uprightness_thresh

    Cells with fewer than `min_pts_per_cell` are kept untouched
    (we don't want to penalise a sparse far-range patch).

    Returns
    -------
    kept     : (M, 3) ground points that survived
    dropped  : (K, 3) points in rejected cells (for visualisation)
    stats    : dict of per-cell metrics (cx, cy, sigma, |n_z|, n_pts, kept_flag)
    """
    g = ground_points.astype(np.float64, copy=False)

    x_edges = np.arange(x_range[0], x_range[1] + cell_size, cell_size)
    y_edges = np.arange(y_range[0], y_range[1] + cell_size, cell_size)

    ix = np.digitize(g[:, 0], x_edges) - 1
    iy = np.digitize(g[:, 1], y_edges) - 1

    valid = (ix >= 0) & (ix < len(x_edges) - 1) & (iy >= 0) & (iy < len(y_edges) - 1)
    out_of_grid = g[~valid]   # outside ROI -- keep unchanged
    g_in   = g[valid]
    ix, iy = ix[valid], iy[valid]

    lin = ix * len(y_edges) + iy

    keep_mask = np.ones(len(g_in), dtype=bool)
    cxs, cys, sigmas, nzs, npts, flags = [], [], [], [], [], []

    for lin_key in np.unique(lin):
        cell_idx = np.where(lin == lin_key)[0]
        cell_pts = g_in[cell_idx]
        i_cell = lin_key // len(y_edges)
        j_cell = lin_key %  len(y_edges)
        cx = 0.5 * (x_edges[i_cell] + x_edges[i_cell + 1])
        cy = 0.5 * (y_edges[j_cell] + y_edges[j_cell + 1])

        if len(cell_pts) < min_pts_per_cell:
            cxs.append(cx); cys.append(cy)
            sigmas.append(np.nan); nzs.append(np.nan)
            npts.append(len(cell_pts)); flags.append(True)   # kept (sparse)
            continue

        n, _, eigvals = _pca_plane(cell_pts)
        sigma = float(eigvals[2] / (eigvals.sum() + 1e-12))
        nz    = float(abs(n[2]))

        keep = (sigma <= flatness_thresh) and (nz >= uprightness_thresh)
        if not keep:
            keep_mask[cell_idx] = False

        cxs.append(cx); cys.append(cy)
        sigmas.append(sigma); nzs.append(nz)
        npts.append(len(cell_pts)); flags.append(keep)

    kept_in    = g_in[keep_mask]
    dropped_in = g_in[~keep_mask]
    kept       = np.concatenate([kept_in, out_of_grid], axis=0)
    dropped    = dropped_in    # out-of-grid never dropped

    stats = dict(
        cx=np.array(cxs), cy=np.array(cys),
        sigma=np.array(sigmas), nz=np.array(nzs),
        n_pts=np.array(npts), kept=np.array(flags),
    )
    return kept, dropped, stats


def plot_eigen_flatness_diagnostic(stats, cell_size, flatness_thresh,
                                   uprightness_thresh):
    """Three-panel diagnostic of the post-filter."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    cm_kept = np.where(stats["kept"], 1.0, 0.0)
    sc0 = axes[0].scatter(stats["cx"], stats["cy"], c=cm_kept,
                          cmap="RdYlGn", s=(cell_size * 9) ** 2, marker="s",
                          vmin=0, vmax=1)
    axes[0].plot(0, 0, "o", color="dodgerblue", ms=10)
    axes[0].set_title(f"Kept (green) vs dropped (red)\ncells")
    plt.colorbar(sc0, ax=axes[0], shrink=0.7, ticks=[0, 1],
                 label="kept (1) / dropped (0)")

    sc1 = axes[1].scatter(stats["cx"], stats["cy"], c=stats["sigma"],
                          cmap="magma_r", s=(cell_size * 9) ** 2, marker="s")
    axes[1].plot(0, 0, "o", color="dodgerblue", ms=10)
    axes[1].set_title(r"Flatness  $\sigma = \lambda_3 / \sum\lambda_i$"
                      + f"\n(threshold = {flatness_thresh})")
    plt.colorbar(sc1, ax=axes[1], shrink=0.7, label="sigma")

    sc2 = axes[2].scatter(stats["cx"], stats["cy"], c=stats["nz"],
                          cmap="viridis", s=(cell_size * 9) ** 2, marker="s",
                          vmin=0.5, vmax=1.0)
    axes[2].plot(0, 0, "o", color="dodgerblue", ms=10)
    axes[2].set_title(r"Uprightness $|n_z|$"
                      + f"\n(threshold = {uprightness_thresh})")
    plt.colorbar(sc2, ax=axes[2], shrink=0.7, label="|n_z|")

    for ax in axes:
        ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")
        ax.set_aspect("equal"); ax.grid(True, lw=0.3)
    plt.tight_layout(); plt.show()


# ============================================================
# Run the optional filter (or skip it)
# ============================================================
if APPLY_EIGEN_FLATNESS_FILTER:
    print(f"[GLE] Applying eigenvalue+flatness post-filter "
          f"(cell={POST_CELL_SIZE} m, sigma <= {FLATNESS_THRESHOLD}, "
          f"|n_z| >= {UPRIGHTNESS_THRESHOLD})")

    ground, dropped, stats = eigen_flatness_filter(
        ground_pw,
        cell_size=POST_CELL_SIZE,
        min_pts_per_cell=POST_MIN_PTS_PER_CELL,
        flatness_thresh=FLATNESS_THRESHOLD,
        uprightness_thresh=UPRIGHTNESS_THRESHOLD,
    )
    n_kept_cells    = int(stats["kept"].sum())
    n_dropped_cells = int((~stats["kept"]).sum())
    print(f"[GLE] Cells kept:    {n_kept_cells}")
    print(f"[GLE] Cells dropped: {n_dropped_cells}")
    print(f"[GLE] Points kept:   {len(ground):,}  (was {len(ground_pw):,})")
    print(f"[GLE] Points dropped:{len(dropped):,}")

    plot_eigen_flatness_diagnostic(stats, POST_CELL_SIZE,
                                   FLATNESS_THRESHOLD, UPRIGHTNESS_THRESHOLD)

    # Side-by-side: PW++-only ground vs GLE-filtered ground
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].scatter(ground_pw[:, 0], ground_pw[:, 1], c=ground_pw[:, 2],
                    cmap="terrain", s=1)
    axes[0].set_title(f"Patchwork++ ground only ({len(ground_pw):,} pts)")
    axes[1].scatter(ground[:, 0], ground[:, 1], c=ground[:, 2],
                    cmap="terrain", s=1)
    if len(dropped):
        axes[1].scatter(dropped[:, 0], dropped[:, 1], c="crimson", s=1,
                        alpha=0.4, label=f"dropped ({len(dropped):,})")
        axes[1].legend(markerscale=5, loc="upper right")
    axes[1].set_title(f"After GLE post-filter ({len(ground):,} pts)")
    for ax in axes:
        ax.set_xlim(-40, 40); ax.set_ylim(-40, 40)
        ax.set_aspect("equal"); ax.grid(True, lw=0.3)
        ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")
    plt.tight_layout(); plt.show()

else:
    print("[GLE] APPLY_EIGEN_FLATNESS_FILTER = False -- "
          "passing Patchwork++ output through unchanged.")
    ground_pw=ground

## 5. Export the filtered ground points



In [ ]:
def save_as_nuscenes_bin(points_xyz, path, intensity=0.0, ring=0):
    """Save (N, 3) XYZ as nuScenes-format .bin -- (N, 5) float32 with zero
    intensity and ring padding."""
    n = len(points_xyz)
    arr = np.zeros((n, 5), dtype=np.float32)
    arr[:, :3] = points_xyz.astype(np.float32)
    arr[:, 3]  = intensity
    arr[:, 4]  = ring
    arr.tofile(path)
    print(f"Saved {n:,} points  ->  {path}  ({arr.nbytes / 1024:.1f} KB)")

In [ ]:
import datetime

_ts = datetime.datetime.fromtimestamp(MCAP_FRAME_TIMESTAMP)

# Round down to nearest 5-minute interval (14:42 → 14:40, 14:47 → 14:45)
rounded_minute = (_ts.minute // 5) * 5
date_folder = _ts.strftime("%Y_%m_%d")                          # "2026_05_01"
time_folder = f"{_ts.strftime('%H')}_{rounded_minute:02d}_00"  # "14_40_00"

# Unix timestamp as integer (for traceability back to exact moment)
unix_t = int(MCAP_FRAME_TIMESTAMP)

if EXPORT_GROUND:
    suffix   = "_pw_gle" if APPLY_EIGEN_FLATNESS_FILTER else "_pw"
    bin_name = f"{GROUND_BIN_BASE}{suffix}_{date_folder}_{time_folder}_{unix_t}.bin"
    out_path = os.path.join(GROUND_BIN_DIR, bin_name)

    os.makedirs(GROUND_BIN_DIR, exist_ok=True)
    save_as_nuscenes_bin(ground, out_path)

    reloaded = load_nuscenes_bin(out_path)
    assert len(reloaded) == len(ground), "Round-trip count mismatch!"
    print(f"Timestamp encoded  : {date_folder}  {time_folder}  (unix={unix_t})")
    print(f"Round-trip OK      : {len(reloaded):,} points read back from\n  {out_path}")
    print(f"\n>>> Notebook 2: set INPUT_BIN_PATH to:\n    {out_path}")
else:
    print("EXPORT_GROUND = False, skipping export.")